# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Outcome B (code-only): URC dataset and trained RiceNet checkpoint were distributed only via Baidu Pan shares that now return '百度网盘-链接不存在' (verified 2026-10-07, page titles only; no dataset bytes on DGX). Author repo issues #10-#18 confirm expiry with no replacement; no Zenodo/Kaggle/HF/GitHub mirror exists. Faithful inference impossible; code-only notebook delivered with RuntimeError guard, statically validated, never executed.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Rice Plant Counting, Locating, and Sizing Method Based on High-Throughput UAV RGB Images

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-49b9b66bc2a39eae0216b71a44fbfcde)


# RiceNet — Rice plant counting from UAV RGB images (CODE-ONLY / NOT EXECUTED)

**Paper**: Wang, A. et al. "Rice Plant Counting, Locating, and Sizing Method Based on High-Throughput UAV RGB Images."
*Plant Phenomics* **5**, 0020 (2023). DOI: [10.34133/plantphenomics.0020](https://doi.org/10.34133/plantphenomics.0020) — open access (PMC10076056, CC BY 4.0).

**Author code**: [`xdbai-source/Rice-Plant-Counting`](https://github.com/xdbai-source/Rice-Plant-Counting) pinned at commit `49c2db62b92e85afd681c4536e8b49714067a37c`.

**PhenoPaper record**: `p-49b9b66bc2a39eae0216b71a44fbfcde` · investigation `p-49b9b66bc2a39eae0216b71a44fbfcde-20260930T133457Z-3695305`.

---

## ⛔ This notebook is a code-only reference — do not expect results

**English**: The essential inputs — the **URC dataset** (355 images, 257,793 point labels; test split 109 images) and the **trained RiceNet checkpoint** (`checkpoint_best.pth`) — were shared only via Baidu Pan links in the author README. Both share links were checked on **2026-10-07** and return the Baidu page *“百度网盘-链接不存在”* (link does not exist). Nine GitHub issues on the author repository (#10–#18, Oct 2023 – Oct 2024) independently report the links as expired, with no replacement published; no Zenodo, Kaggle, Hugging Face, or GitHub mirror was found. The paper's Data Availability statement says the data are "available on request". A faithful inference demonstration is therefore **impossible without the authors providing a new access route**; this notebook intentionally contains **no executed outputs**.

**日本語**: 本ノートブックは**コードのみ（未実行）**の参考資料です。必須入力（URCデータセットと学習済みチェックポイント）はBaidu Pan専用で共有されており、2026-10-07時点でリンク無効（“百度网盘-链接不存在”）を確認しました。著者リポジトリのIssue #10〜#18（2023-10〜2024-10）もリンク切れを報告しており、代替の公開ミラー（Zenodo/Kaggle/Hugging Face/GitHub）は見つかりませんでした。論文のData Availabilityは「オンリクエストで利用可能」としています。よって忠実な推論デモは実行できず、出力は意図的に空です。

## 1. Scope of this reference implementation

**English**: This notebook preserves the author's *counting evaluation* path (`eval.py`) as runnable-once-materials-exist code: URC test split loading, RiceNet inference, and MAE/RMSE against the paper's reported **MAE 8.6 / RMSE 11.2** (Table 1). The locate/size path (`detect.py`) additionally requires `size_pth/*.pth` checkpoints and `tep_density` labels that were never present in the shared dataset (author issue #17), so it is documented but out of scope. The first code cell below is a **hard guard**: it stops "Run all" so the notebook can never produce a misleading result.

**日本語**: 本ノートブックは著者のカウント評価経路（`eval.py`）を、材料が入手できた場合に実行可能な形で保存します。位置・サイズ推定（`detect.py`）は共有データセットに存在しない追加ファイルを要求するため対象外です。最初のコードセルはハードガードで、"Run all" を停止させます。

In [ ]:
# GUARD — outcome B (essential material missing). This notebook must not execute.
raise RuntimeError(
    "CODE-ONLY / NOT EXECUTED: the URC dataset and the trained RiceNet "
    "checkpoint were only distributed via Baidu Pan links that are now dead "
    "(verified 2026-10-07: share pages return '百度网盘-链接不存在'; author "
    "repo issues #10-#18 report expiry; no public mirror found). See the "
    "markdown cells and the referenced report for evidence. Do not run this "
    "notebook unless you obtain the materials through a legitimate route."
)

### 1.1 Environment the author's code assumes

**English**: `eval.py`/`dataset.py` need PyTorch, torchvision, h5py, OpenCV, matplotlib, and scipy (imported at module level in the pinned files). The author's `eval.py` hard-codes `torch.device('cuda:0')`; a CPU fallback is a one-line adaptation shown later. The VGG16-bn frontend weights are downloaded automatically by `models.py` from `download.pytorch.org` — the *decoder* weights, however, come only from the dead Baidu checkpoint.

**日本語**: 著者コードは PyTorch / torchvision / h5py / OpenCV / matplotlib / scipy を要求します。`eval.py` は `cuda:0` をハードコードしています（後段でCPUフォールバックを提示）。VGG16-bn の前端重みは自動ダウンロードされますが、デコーダ重みはBaidu Panのチェックポイントにのみ存在します。

In [ ]:
# Dependency install (would run only if the guard above is removed on purpose).
# Versions chosen for current Colab compatibility; not verified here because
# this notebook is code-only by policy.
%pip install -q torch torchvision h5py opencv-python-headless scipy matplotlib

## 2. Obtaining the essential materials — the verified blocker

**English**: The author README (pinned commit `49c2db62`, step 3) links the URC dataset and the checkpoint only via Baidu Pan:

- dataset: `https://pan.baidu.com/s/1L2g6HMjjY7kaJmGn9HTNzA?pwd=g1rq` (code `g1rq`) — **dead** as of 2026-10-07;
- checkpoint: `https://pan.baidu.com/s/1o-X2wrvZTTmfxGbfWfUfsQ?pwd=1w3y` (code `1w3y`) — **dead** as of 2026-10-07.

These URLs are recorded as **evidence of the blocker**, not as a download source. Baidu Pan shares also have no documented anonymous download API that a learner notebook could rely on: even a live share requires the client web app with an account. The cell below therefore fails explicitly if the expected files are absent, instead of inventing a URL.

**日本語**: 著者READMEのBaidu Panリンク2件は2026-10-07時点で無効です。ここではURLをブロッカーの証拠として記録するだけであり、代替URLを捏造しません。以下のセルは、期待するファイルが存在しない場合に明示的に失敗します。

In [ ]:
# Material availability check (faithful to the author's expected layout).
# Expected after unpacking both shares into the working directory:
#   ./URC/test/imgs_4/*.jpg        (109 test images, 1368x912)
#   ./URC/test/new_data_4/*.h5     (density, attention, kpoint arrays)
#   ./URC/test/dis_data_4/*.h5     (plant size 'dis' labels)
#   ./checkpoint/RiceNet/checkpoint_best.pth   (trained RiceNet state dict)
import os
from pathlib import Path

MATERIAL_STATUS = {
    "URC test images": Path("URC/test/imgs_4"),
    "URC test labels": Path("URC/test/new_data_4"),
    "URC dis labels": Path("URC/test/dis_data_4"),
    "RiceNet checkpoint": Path("checkpoint/RiceNet/checkpoint_best.pth"),
}
missing = [k for k, p in MATERIAL_STATUS.items() if not p.exists()]
if missing:
    raise RuntimeError(
        f"Essential materials missing: {missing}. "
        "The only author-provided source is the now-dead Baidu Pan share "
        "(see notebook header). No legitimate public mirror was found as of "
        "2026-10-07. Request the data from the authors or supply your own copy."
    )
print("All essential materials present.")

## 3. RiceNet architecture (pinned author code)

**English**: The cell below retrieves the author's `models.py` and `dataset.py` **from the pinned GitHub commit** into the runtime and imports them — this keeps the network definition bit-identical to the paper instead of re-typing it. `Model` is a VGG16-bn frontend feeding two `BackEnd` decoders: the attention branch output passes through a sigmoid (`conv_att`), multiplies the density branch, and a final 1×1 conv (`conv_out`) yields the density map at half input resolution. `load_vgg()` maps ImageNet VGG16-bn features into the custom `BaseConv` layers by name. Frontend weights come from `download.pytorch.org`; decoder weights come from the checkpoint loaded in section 4.

**日本語**: 次セルはピン留めしたコミットから著者の `models.py` / `dataset.py` を取得してインポートし、ネットワーク定義を論理どおりに保ちます。RiceNet は VGG16-bn 前端＋2つの `BackEnd` デコーダで、注意分岐を sigmoid で乗算して密度マップを得ます。前端重みは PyTorch 公式、デコーダ重みはセクション4のチェックポイントから読み込みます。

In [ ]:
# Fetch the pinned author modules INSIDE the Colab runtime (code only).
# Pin: xdbai-source/Rice-Plant-Counting @ 49c2db62b92e85afd681c4536e8b49714067a37c
import urllib.request
from pathlib import Path

PINNED = "https://raw.githubusercontent.com/xdbai-source/Rice-Plant-Counting/49c2db62b92e85afd681c4536e8b49714067a37c"
for fname in ("models.py", "dataset.py", "transforms.py"):
    dest = Path("/content") / fname
    urllib.request.urlretrieve(f"{PINNED}/{fname}", dest)
    print(f"fetched {fname} ({dest.stat().st_size} bytes) from pinned commit")

import sys
sys.path.insert(0, "/content")
from models import Model  # noqa: E402  (author's RiceNet definition)
import dataset  # noqa: E402  (author's Dataset class)

## 4. Counting evaluation on the URC test split (port of `eval.py`)

**English**: This is the author's evaluation loop, preserved with only two documented adaptations: (1) device selection falls back to CPU when CUDA is absent (`eval.py` hard-codes `cuda:0`); (2) `torch.load` uses `map_location` for the same reason. Everything else — batch size 1, the `Dataset('rice', is_train=False)` path (`test/imgs_4` + `test/new_data_4`, images resized to multiples of 16 and normalized with ImageNet statistics), `model(images)` returning the density map, and counting as `predict.sum()` — is exactly as in the pinned `eval.py`. Reference values from the paper (Table 1): **MAE 8.6, RMSE 11.2** over the 109 test images. This notebook has not produced these numbers; they are quoted for comparison only.

**日本語**: 著者の `eval.py` の評価ループを、(1) CPUフォールバック、(2) `map_location` の2点の変更のみで再現します。バッチ1・データセット経路・前処理・`predict.sum()` による計数は原典どおりです。論文Table 1の参照値（MAE 8.6 / RMSE 11.2、109枚）は引用のみで、本ノートブックはまだ何も計算していません。

In [ ]:
# Counting evaluation over the URC test split (faithful port of author eval.py).
import torch
from torch.utils import data

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # ADAPTATION 1 (author: cuda:0)
print(f"device: {device}")

test_dataset = dataset.Dataset("URC", "rice", False)
test_loader = data.DataLoader(test_dataset, batch_size=1, shuffle=False)

model = Model().to(device)
state = torch.load("checkpoint/RiceNet/checkpoint_best.pth", map_location=device)  # ADAPTATION 2 (map_location)
model.load_state_dict(state)  # author's eval.py: the loaded object is the raw state dict
model.eval()

mae, mse = 0.0, 0.0
with torch.no_grad():
    for images, gt, _dis, _path in test_loader:
        images, gt = images.to(device), gt.to(device)
        predict, _ = model(images)
        mae += torch.abs(predict.sum() - gt).item()
        mse += ((predict.sum() - gt) ** 2).item()
        print(f"predict:{predict.sum().item():.2f} label:{gt.item():.2f}")
mae /= len(test_loader)
rmse = (mse / len(test_loader)) ** 0.5
print(f"MAE: {mae:.2f}  RMSE: {rmse:.2f}   (paper Table 1: 8.6 / 11.2)")

### 4.1 Sanity check against the paper

**English**: If the loop above ever runs with the genuine materials, compare its MAE/RMSE with Table 1 (MAE 8.6, RMSE 11.2). The author's `eval.py` additionally computes relative MAE/RMSE and dumps per-image predictions to `./n_2.pth`; a small table of per-image predicted vs. labeled counts plus a density-map visualization (input image, predicted density, attention map) would be the minimal readable output. None of that can be shown here because the inputs do not exist on any authorized public route.

**日本語**: 正規の材料が入手できた場合は、Table 1（MAE 8.6 / RMSE 11.2）と比較してください。著者コードは相対MAE/RMSEも計算し、予測を `n_2.pth` に保存します。入力が存在しないため、ここに表示できる結果はありません。

## 5. Out of scope: locating and sizing (`detect.py`)

**English**: The paper's plant locating (density-map peak detection with the LNMS rule) and sizing (KD-tree nearest-neighbor mean × 0.8 in `tool.py`) are preserved in the pinned repository, but running them needs the `size_pth/checkpoint_bestdis.pth` and `checkpoint_latestdis.pth` checkpoints (same dead Baidu share) and `tep_density` labels that the shared URC dataset does not contain — a defect the author was told about in issue #17 without a fix. `detect.py` also depends on `visdom` for plotting. All of this is documented but intentionally not ported.

**日本語**: 位置・サイズ推定（`detect.py` + `tool.py`）は pinned リポジトリに保存されていますが、同じく無効な共有リンクの `size_pth` チェックポイントと、共有データセットに存在しない `tep_density` ラベル（Issue #17で報告、未修正）を必要とするため、対象外としました。

## 6. Verification record and references

**English**:
- **Not executed.** No Colab runtime was allocated for this notebook; all code cells are unrun and outputs are empty by policy (AGENTS.md outcome B). The only execution attempted in this run was the dead-link verification above, performed by the authoring agent from pinned repository metadata and Baidu share *page* titles — no dataset bytes were ever handled on the host.
- **Evidence**: share-page titles “百度网盘-链接不存在” for both links (2026-10-07); author repo issues #10–#18; author's last link update 2023-04-08 (issue #6); negative searches on Zenodo, Kaggle, Hugging Face, and GitHub code search (share IDs).
- **Pinned sources**: `xdbai-source/Rice-Plant-Counting` @ `49c2db62b92e85afd681c4536e8b49714067a37c` — `eval.py`, `models.py`, `dataset.py`, `tool.py`.
- **Paper**: Plant Phenomics 5:0020 (2023), DOI 10.34133/plantphenomics.0020 (PMC10076056, CC BY 4.0). Flow record: investigation `p-49b9b66bc2a39eae0216b71a44fbfcde-20260930T133457Z-3695305` (AI-generated guidance, cross-checked against primary code).
- **Next step if materials appear**: obtain the authors' permission-based copy, verify the checkpoint loads, remove the guard cell intentionally, and re-run the whole notebook top-to-bottom in a fresh runtime.

**日本語**: 本ノートブックは未実行です（ポリシー上、出力は意図的に空）。リンク切れの検証のみがこのrunで実施されました。材料が入手できた場合は、ガードセルを意図的に削除してから新しいランタイムで全体を再実行してください。